# EXP-13: Multi-Seed Three-SVC Ensemble

**Competition:** SPR 2026 — Mammography Report Classification  
**Base:** V12 notebook (0.80403) — Three-SVC + LGB  
**Enhancement:** 5 seeds per model → 20 models total, averaged for robustness  
**Runtime:** CPU only  

**Estratégia:**
- Mesmo pipeline do V12 (3 SVCs + 1 LGB) mas com 5 seeds cada
- 20 modelos totais → média de probabilidades reduz variância
- GroupKFold 5-fold CV para validação local confiável
- Threshold optimization usando OOF predictions
- Clinical guardrails mantidos

In [ ]:
import os, re, hashlib, warnings, time
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.pipeline import FeatureUnion
from sklearn.model_selection import GroupKFold
from sklearn.metrics import f1_score, classification_report
from scipy.sparse import hstack, csr_matrix
import lightgbm as lgb
warnings.filterwarnings('ignore')

TRAIN_PATH = '/kaggle/input/competitions/spr-2026-mammography-report-classification/train.csv'
TEST_PATH  = '/kaggle/input/competitions/spr-2026-mammography-report-classification/test.csv'

train = pd.read_csv(TRAIN_PATH)
test  = pd.read_csv(TEST_PATH)

N_FOLDS = 5
N_CLASSES = 7
SEEDS = [42, 123, 456, 789, 2026]

print(f'Train: {train.shape}, Test: {test.shape}')
print(f'Class distribution:\n{train["target"].value_counts().sort_index()}')

In [ ]:
def clean_achados(s: str) -> str:
    if pd.isna(s): return ""
    s = str(s).strip().lower()
    s = s.replace("\r\n", "\n").replace("\r", "\n")
    s = re.sub(r"[ \t]+", " ", s)
    s = re.sub(r"\n{2,}", "\n", s)
    match = re.search(r'achados:(.*?)(análise comparativa:|$)', s, flags=re.DOTALL)
    if match: s = match.group(1).strip()
    s = re.sub(r'[0-9]+,[0-9]+', 'NUM', s)
    s = re.sub(r'[0-9]+', 'NUM', s)
    return s

def clean_full(s: str) -> str:
    if pd.isna(s): return ""
    s = str(s).strip().lower()
    s = s.replace("\r\n", "\n").replace("\r", "\n")
    s = re.sub(r"[ \t]+", " ", s)
    s = re.sub(r'[0-9]+,[0-9]+', 'NUM', s)
    s = re.sub(r'[0-9]+', 'NUM', s)
    return s

def stable_hash(s: str) -> str:
    return hashlib.md5(s.encode("utf-8")).hexdigest()

def extract_dense_features(df):
    features = pd.DataFrame(index=df.index)
    text_col = df['report'].fillna('').astype(str).str.lower()
    features['report_length']   = text_col.apply(len)
    features['has_measurement'] = text_col.str.contains(r'\b(?:cm|mm|medindo)\b', regex=True).astype(int)
    features['has_spiculation'] = text_col.str.contains(r'espiculad', regex=True).astype(int)
    features['has_distortion']  = text_col.str.contains(r'distorção arquitetural', regex=True).astype(int)
    features['has_biopsy']      = text_col.str.contains(r'biopsy|biópsia|resultado de cine|carcinoma', regex=True).astype(int)
    return csr_matrix(features.values)

train["achados"] = train["report"].apply(clean_achados)
train["full"]    = train["report"].apply(clean_full)
test["achados"]  = test["report"].apply(clean_achados)
test["full"]     = test["report"].apply(clean_full)
train["group"]   = train["report"].apply(stable_hash)

X_train_dense = extract_dense_features(train)
X_test_dense  = extract_dense_features(test)
y = train["target"].astype(int).values
groups = train["group"].values

print("Preprocessing done.")

In [ ]:
print("Building TF-IDF features...")

# Achados TF-IDF (same as V12)
tfidf_A = FeatureUnion([
    ("word", TfidfVectorizer(ngram_range=(1, 3), min_df=3, max_df=0.95, sublinear_tf=True)),
    ("char", TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=3, max_df=0.95,
                             sublinear_tf=True, max_features=80000))
])
X_train_A = tfidf_A.fit_transform(train["achados"].values)
X_test_A  = tfidf_A.transform(test["achados"].values)

# Full text TF-IDF config 1
tfidf_F = FeatureUnion([
    ("word", TfidfVectorizer(ngram_range=(1, 3), min_df=3, max_df=0.95, sublinear_tf=True)),
    ("char", TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=3, max_df=0.95,
                             sublinear_tf=True, max_features=80000))
])
X_train_F = tfidf_F.fit_transform(train["full"].values)
X_test_F  = tfidf_F.transform(test["full"].values)

# Full text TF-IDF config 2 (wider char ngrams)
tfidf_F2 = FeatureUnion([
    ("word", TfidfVectorizer(ngram_range=(1, 3), min_df=3, max_df=0.95, sublinear_tf=True)),
    ("char", TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 6), min_df=3, max_df=0.95,
                             sublinear_tf=True, max_features=100000))
])
X_train_F2 = tfidf_F2.fit_transform(train["full"].values)
X_test_F2  = tfidf_F2.transform(test["full"].values)

# LGB features: achados + dense
X_train_lgb = hstack([X_train_A, X_train_dense]).tocsr()
X_test_lgb  = hstack([X_test_A,  X_test_dense]).tocsr()

print(f"SVC-A: {X_train_A.shape[1]}, SVC-F: {X_train_F.shape[1]}, SVC-F2: {X_train_F2.shape[1]}, LGB: {X_train_lgb.shape[1]}")

In [ ]:
# =============================================================================
# Multi-Seed GroupKFold CV — 4 model types × 5 seeds = 20 models
# =============================================================================

gkf = GroupKFold(n_splits=N_FOLDS)
folds = list(gkf.split(X_train_A, y, groups))

oof_probas = {}   # model_name -> (n_train, 7)
test_probas = {}  # model_name -> (n_test, 7)

t0 = time.time()

for seed in SEEDS:
    # --- SVC-A ---
    name = f'svc_A_s{seed}'
    print(f"\n--- {name} ---")
    oof = np.zeros((len(train), N_CLASSES), dtype=np.float64)
    te_acc = np.zeros((len(test), N_CLASSES), dtype=np.float64)
    for fi, (tr_idx, va_idx) in enumerate(folds):
        m = CalibratedClassifierCV(LinearSVC(class_weight="balanced", random_state=seed, max_iter=10000), cv=3, method='sigmoid')
        m.fit(X_train_A[tr_idx], y[tr_idx])
        oof[va_idx] = m.predict_proba(X_train_A[va_idx])
        te_acc += m.predict_proba(X_test_A) / N_FOLDS
    print(f"  OOF F1: {f1_score(y, np.argmax(oof, axis=1), average='macro'):.4f}")
    oof_probas[name] = oof
    test_probas[name] = te_acc

    # --- SVC-F ---
    name = f'svc_F_s{seed}'
    print(f"--- {name} ---")
    oof = np.zeros((len(train), N_CLASSES), dtype=np.float64)
    te_acc = np.zeros((len(test), N_CLASSES), dtype=np.float64)
    for fi, (tr_idx, va_idx) in enumerate(folds):
        m = CalibratedClassifierCV(LinearSVC(class_weight="balanced", random_state=seed, max_iter=10000), cv=3, method='sigmoid')
        m.fit(X_train_F[tr_idx], y[tr_idx])
        oof[va_idx] = m.predict_proba(X_train_F[va_idx])
        te_acc += m.predict_proba(X_test_F) / N_FOLDS
    print(f"  OOF F1: {f1_score(y, np.argmax(oof, axis=1), average='macro'):.4f}")
    oof_probas[name] = oof
    test_probas[name] = te_acc

    # --- SVC-F2 ---
    name = f'svc_F2_s{seed}'
    print(f"--- {name} ---")
    oof = np.zeros((len(train), N_CLASSES), dtype=np.float64)
    te_acc = np.zeros((len(test), N_CLASSES), dtype=np.float64)
    for fi, (tr_idx, va_idx) in enumerate(folds):
        m = CalibratedClassifierCV(LinearSVC(class_weight="balanced", random_state=seed, max_iter=10000), cv=3, method='sigmoid')
        m.fit(X_train_F2[tr_idx], y[tr_idx])
        oof[va_idx] = m.predict_proba(X_train_F2[va_idx])
        te_acc += m.predict_proba(X_test_F2) / N_FOLDS
    print(f"  OOF F1: {f1_score(y, np.argmax(oof, axis=1), average='macro'):.4f}")
    oof_probas[name] = oof
    test_probas[name] = te_acc

    # --- LGB ---
    name = f'lgb_s{seed}'
    print(f"--- {name} ---")
    oof = np.zeros((len(train), N_CLASSES), dtype=np.float64)
    te_acc = np.zeros((len(test), N_CLASSES), dtype=np.float64)
    for fi, (tr_idx, va_idx) in enumerate(folds):
        m = lgb.LGBMClassifier(class_weight='balanced', n_estimators=300, learning_rate=0.05,
                               max_depth=6, random_state=seed, n_jobs=-1, verbose=-1)
        m.fit(X_train_lgb[tr_idx], y[tr_idx])
        oof[va_idx] = m.predict_proba(X_train_lgb[va_idx])
        te_acc += m.predict_proba(X_test_lgb) / N_FOLDS
    print(f"  OOF F1: {f1_score(y, np.argmax(oof, axis=1), average='macro'):.4f}")
    oof_probas[name] = oof
    test_probas[name] = te_acc

elapsed = time.time() - t0
print(f"\n{'='*60}")
print(f"All {len(oof_probas)} models trained in {elapsed:.0f}s")

In [ ]:
# =============================================================================
# Ensemble: average across seeds, then combine with V12 weights
# =============================================================================

# Average OOF probas per model type across seeds
def avg_across_seeds(probas_dict, prefix):
    matching = {k: v for k, v in probas_dict.items() if k.startswith(prefix)}
    return np.mean(list(matching.values()), axis=0)

# OOF
oof_svc_A  = avg_across_seeds(oof_probas, 'svc_A')
oof_svc_F  = avg_across_seeds(oof_probas, 'svc_F_')
oof_svc_F2 = avg_across_seeds(oof_probas, 'svc_F2')
oof_lgb    = avg_across_seeds(oof_probas, 'lgb')

# Test
te_svc_A  = avg_across_seeds(test_probas, 'svc_A')
te_svc_F  = avg_across_seeds(test_probas, 'svc_F_')
te_svc_F2 = avg_across_seeds(test_probas, 'svc_F2')
te_lgb    = avg_across_seeds(test_probas, 'lgb')

# V12 weights: SVC ensemble = 0.25*A + 0.40*F + 0.35*F2, then 0.70*SVC + 0.30*LGB
oof_svc_ens = 0.25 * oof_svc_A + 0.40 * oof_svc_F + 0.35 * oof_svc_F2
oof_blend   = 0.70 * oof_svc_ens + 0.30 * oof_lgb

te_svc_ens = 0.25 * te_svc_A + 0.40 * te_svc_F + 0.35 * te_svc_F2
te_blend   = 0.70 * te_svc_ens + 0.30 * te_lgb

# Baseline OOF score (no thresholds)
baseline_preds = np.argmax(oof_blend, axis=1)
baseline_f1 = f1_score(y, baseline_preds, average='macro')
print(f"OOF macro-F1 (no thresholds): {baseline_f1:.4f}")
print(f"\nPer-class OOF F1:")
print(classification_report(y, baseline_preds, digits=4))

In [ ]:
# =============================================================================
# OOF-Optimized Threshold Search for rare classes
# =============================================================================
# Priority order: class 6 (most severe) → 5 → 4 → 3
# Search thresholds that maximize OOF macro-F1

threshold_classes = [6, 5, 4, 3]
threshold_range = np.arange(0.05, 0.55, 0.01)

def apply_thresholds(proba, thresholds):
    preds = np.argmax(proba, axis=1).copy()
    for cls in [6, 5, 4, 3]:
        if cls in thresholds:
            mask = proba[:, cls] > thresholds[cls]
            for higher_cls in [c for c in [6, 5, 4, 3] if c != cls]:
                if higher_cls in thresholds and threshold_classes.index(higher_cls) < threshold_classes.index(cls):
                    mask = mask & (preds != higher_cls)
            preds[mask] = cls
    return preds

best_thresholds = {}
current_f1 = baseline_f1

for cls in threshold_classes:
    best_cls_f1 = current_f1
    best_cls_thr = None
    for thr in threshold_range:
        trial = {**best_thresholds, cls: thr}
        trial_preds = apply_thresholds(oof_blend, trial)
        trial_f1 = f1_score(y, trial_preds, average='macro')
        if trial_f1 > best_cls_f1:
            best_cls_f1 = trial_f1
            best_cls_thr = thr
    if best_cls_thr is not None:
        best_thresholds[cls] = best_cls_thr
        current_f1 = best_cls_f1
        print(f"Class {cls}: threshold={best_cls_thr:.2f}, macro-F1={best_cls_f1:.4f}")
    else:
        print(f"Class {cls}: no improvement from thresholds")

final_oof_preds = apply_thresholds(oof_blend, best_thresholds)
final_oof_f1 = f1_score(y, final_oof_preds, average='macro')
print(f"\nFinal OOF macro-F1 with thresholds: {final_oof_f1:.4f}")
print(f"Optimal thresholds: {best_thresholds}")
print(f"\nPer-class F1 with thresholds:")
print(classification_report(y, final_oof_preds, digits=4))

In [ ]:
# =============================================================================
# Apply thresholds + clinical guardrails → submission
# =============================================================================

test_preds = apply_thresholds(te_blend, best_thresholds)

def apply_safe_rules(row):
    text = str(row['report']).lower()
    current_pred = int(row['target'])
    if re.search(r'resultado de cine grau 3|carcinoma|\bcdis\b', text):
        return 6
    if 'espiculad' in text and 'distorção' in text and current_pred < 4:
        return 5
    return current_pred

test['target'] = test_preds
test['target'] = test.apply(apply_safe_rules, axis=1)

submission = test[['ID', 'target']].copy()
submission['target'] = submission['target'].astype(int)
submission.to_csv('submission.csv', index=False)

print('Prediction distribution:')
print(submission['target'].value_counts().sort_index())
print(f'\nSubmission saved! Shape: {submission.shape}')
print(submission)